# 02 — Baseline: Logistic Regression

**Why baseline first?** Every serious ML project starts with a simple, interpretable model. Three reasons:

1. **Sanity check**: if logistic regression already hits AUC 0.80, the problem is easy and more complex models will give marginal gains.
2. **Benchmark**: XGBoost must beat this number with margin later.
3. **Interpretability**: coefficients show the sign and magnitude of each feature, validating that the model is learning plausible relationships.

**Strategy**:
- sklearn Pipeline: `SimpleImputer(median)` → `StandardScaler` → `LogisticRegression`
- `class_weight='balanced'` to compensate for the 15:1 class imbalance
- Train on `train`, evaluate on `val` (keep `test` untouched until the end)

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import json

import joblib
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from src.data.split import load_splits
from src.evaluation.metrics import comparison_table, evaluate, plot_evaluation

plt.rcParams['figure.dpi'] = 100

splits = load_splits(Path.cwd().parent / 'data' / 'processed')
train, val, test = splits['train'], splits['val'], splits['test']

FEATURES = [c for c in train.columns if c != 'target']
print(f'Features ({len(FEATURES)}): {FEATURES}')
print(f'\nTrain: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}')

In [ ]:
# Separate X and y
X_train, y_train = train[FEATURES], train['target'].values
X_val,   y_val   = val[FEATURES],   val['target'].values
X_test,  y_test  = test[FEATURES],  test['target'].values

# Convert Float64 (nullable) to float64 — sklearn doesn't handle nullable dtypes well
X_train = X_train.astype({c: 'float64' for c in X_train.columns})
X_val   = X_val.astype({c: 'float64' for c in X_val.columns})
X_test  = X_test.astype({c: 'float64' for c in X_test.columns})

X_train.dtypes

## Pipeline

Imputer → Scaler → LogReg. Three design decisions:

- **Imputer(median)**: `monthly_income` and `dependents` have missing values. Median is robust to outliers (which this dataset has plenty of). Mean would be a bad choice here.
- **StandardScaler**: LogReg with L2 regularization is scale-sensitive. `monthly_income` has std ~5000 while `dependents` has std ~1. Without scaling, regularization penalizes coefficients unfairly.
- **class_weight='balanced'**: automatically adjusts class weights to compensate for imbalance. Without this, the model tends to always predict 'good'.

In [ ]:
pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        max_iter=2000,
        class_weight='balanced',
        random_state=42,
        solver='lbfgs',
    )),
])

pipe.fit(X_train, y_train)
print('Pipeline trained.')

In [ ]:
# Predict probabilities on train and val
proba_train = pipe.predict_proba(X_train)[:, 1]
proba_val   = pipe.predict_proba(X_val)[:, 1]

r_train = evaluate(y_train, proba_train, 'LogReg', 'train')
r_val   = evaluate(y_val,   proba_val,   'LogReg', 'val')

comparison_table([r_train, r_val])

## Quick interpretation

- **AUC train ≈ AUC val**: no serious overfitting. Simple model, generalizes well.
- **KS expected**: 0.40–0.50 on this dataset with balanced logreg.
- **Brier**: decent calibration, can be improved with isotonic calibration later.
- **Precision@base_rate**: how many of the top-6.68% riskiest are actually bad.

In [ ]:
fig = plot_evaluation(y_val, proba_val, 'LogReg — Validation')
plt.show()

## Coefficients — is the model learning plausible things?

Because we standardized the features, coefficients are directly comparable in magnitude.

In [ ]:
import pandas as pd

coefs = (
    pd.DataFrame({'feature': FEATURES, 'coef': pipe.named_steps['clf'].coef_[0]})
    .assign(abs_coef=lambda d: d['coef'].abs())
    .sort_values('abs_coef', ascending=False)
)

coefs[['feature', 'coef']]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#C44E52' if c > 0 else '#4C72B0' for c in coefs['coef']]
ax.barh(coefs['feature'], coefs['coef'], color=colors)
ax.axvline(0, color='black', lw=0.5)
ax.set_xlabel('Standardized coefficient')
ax.set_title('Baseline LogReg — feature importance')
ax.invert_yaxis()
fig.tight_layout()
plt.show()

## Expected results

Coefficients **positive** (increase risk): `past_due_90`, `past_due_30_59`, `past_due_60_89`, `revolving_utilization`, `had_past_due_sentinel`.

Coefficients **negative** (decrease risk): `age`, `monthly_income`, `real_estate_loans`.

This matches credit intuition — delinquency history is the strongest signal, age and income protect. If any coefficient comes out inverted, that's a sign of multicollinearity or a pipeline bug.

In [ ]:
# Save baseline
models_dir = Path.cwd().parent / 'models'
models_dir.mkdir(exist_ok=True)
joblib.dump(pipe, models_dir / 'baseline_logreg.pkl')
print(f'Baseline saved to {models_dir / "baseline_logreg.pkl"}')

# Save metrics for later side-by-side comparison
metrics_log_path = models_dir / 'metrics_log.json'
log = {}
if metrics_log_path.exists():
    log = json.loads(metrics_log_path.read_text())
log['baseline_logreg'] = {
    'train': r_train.to_row(),
    'val': r_val.to_row(),
}
metrics_log_path.write_text(json.dumps(log, indent=2))
print(f'Metrics logged to {metrics_log_path}')